# TSViT + LLM — Clasificacion de Cultivos en PASTIS24

## Arquitectura

```
Encoder (TSViT)                               Decoder (LLM)
                                               Qwen2.5-3B + LoRA
Input: PASTIS24 pickle                    
  img: (T, 10, 24, 24) int16                  Word Embeddings
  doy: (T,) int64                                  |
  labels: (3, 24, 24) uint8                    Masked Multi-Head Attention
       |                                            |
  Normalize + prepend time channel             Feed Forward
  -> (T, 11, 24, 24)                               |
       |                                       Linear -> Softmax
  Patch Embeddings                                  |
       |                                       "Soft winter wheat"
  Temporal Encoder (6 layers, d=128)
  + K=18 CLS tokens
       |
  Spatial Encoder (2 layers, d=128)    Projector (Adapter)
       |                               MLP: 128 -> 2048 -> 2048
  K=18 CLS token outputs (K, 128) --->      |
                                        visual tokens -> LLM
```

### Dataset: PASTIS24
Formato pre-procesado del repo TSViT (DeepSatModels, CVPR 2023).
Los patches 128x128 originales ya vienen divididos en ventanas 24x24.

| Contenido | Shape | Descripcion |
|-----------|-------|-------------|
| `img` | (T, 10, 24, 24) int16 | 10 bandas S2, T fechas |
| `doy` | (T,) int64 | Dia del anio por fecha |
| `labels` | (3, 24, 24) uint8 | Segmentacion semantica |

### Entrenamiento en 2 etapas
| Etapa | Que se entrena | Loss |
|-------|---------------|------|
| Stage 1 | TSViT completo | Focal Loss |
| Stage 2 | Projector + LoRA (TSViT frozen) | Cross-Entropy LM |

### VRAM estimado (RTX 5080 16GB)
| Componente | VRAM |
|-----------|------|
| TSViT encoder (~2M params) | ~0.1 GB |
| Projector MLP (~5M params) | ~0.01 GB |
| Qwen2.5-3B (4-bit + LoRA) | ~3.5 GB |
| Activations + optimizer | ~4-5 GB |
| **Total** | **~8 GB** |

### Fuentes
| Componente | Paper/Repo |
|-----------|------------|
| TSViT | Tarasiou et al., CVPR 2023 — github.com/michaeltrs/DeepSatModels |
| PASTIS | Garnot & Landrieu, 2022 — github.com/VSainteuf/pastis-benchmark |
| Projector | LLaVA, Liu et al. 2023 — arxiv.org/abs/2304.08485 |
| LoRA | Hu et al., ICLR 2022 — arxiv.org/abs/2106.09685 |

In [1]:
# ============================================================
# Celda 1: Instalacion de dependencias (uv)
# ============================================================
import subprocess

deps = [
    "torch", "torchvision", "torchaudio",
    "einops",
    "transformers>=4.45.0", "accelerate", "bitsandbytes", "peft",
    "sentencepiece", "protobuf",
    "numpy", "matplotlib", "tqdm", "scikit-learn", "pandas",
]

for dep in deps:
    subprocess.check_call(["uv", "add", dep])
print("Dependencias instaladas con uv")

Resolved 168 packages in 4ms
Audited 164 packages in 14ms
Resolved 168 packages in 0.54ms
Audited 164 packages in 0.93ms
Resolved 168 packages in 0.62ms
Audited 164 packages in 0.92ms
Resolved 168 packages in 0.57ms
Audited 164 packages in 0.96ms
Resolved 168 packages in 0.54ms
Audited 164 packages in 0.92ms
Resolved 168 packages in 0.49ms
Audited 164 packages in 0.92ms
Resolved 168 packages in 0.51ms
Audited 164 packages in 0.92ms
Resolved 168 packages in 0.57ms
Audited 164 packages in 0.95ms
Resolved 168 packages in 0.52ms
Audited 164 packages in 0.94ms
Resolved 168 packages in 0.57ms
Audited 164 packages in 0.96ms
Resolved 168 packages in 0.52ms
Audited 164 packages in 0.92ms


Dependencias instaladas con uv


Resolved 168 packages in 0.50ms
Audited 164 packages in 0.92ms
Resolved 168 packages in 0.53ms
Audited 164 packages in 0.93ms
Resolved 168 packages in 0.54ms
Audited 164 packages in 0.94ms


In [1]:
# ============================================================
# Celda 2: Imports y configuracion
# ============================================================
import os, pickle, glob, random, math, gc, warnings, json, tempfile
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from einops import rearrange, repeat
from einops.layers.torch import Rearrange
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             classification_report)

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

# -- Paths --
_ROOT = Path.cwd().parent if Path.cwd().name == "reports" else Path.cwd()
PASTIS24_DIR = _ROOT / "dataset" / "PASTIS24"
PICKLE_DIR = PASTIS24_DIR / "pickle24x24"
FOLDS_DIR = PASTIS24_DIR / "fold-paths"

# -- Clases (1-18, same as PASTIS) --
CLASSES = {
    1: "Meadow", 2: "Soft_winter_wheat", 3: "Corn",
    4: "Winter_barley", 5: "Winter_rapeseed", 6: "Spring_barley",
    7: "Sunflower", 8: "Grapevine", 9: "Beet", 10: "Winter_triticale",
    11: "Winter_durum_wheat", 12: "Fruits_vegetables_flowers", 13: "Potatoes",
    14: "Leguminous_fodder", 15: "Soybeans", 16: "Orchard",
    17: "Mixed_cereal", 18: "Sorghum"
}
NUM_CLASSES = 18

# -- Hiperparametros TSViT (CVPR 2023 paper Table 2) --
IMG_RES = 24
PATCH_SIZE = 2
MAX_SEQ_LEN = 60
DIM = 128
TEMPORAL_DEPTH = 6
SPATIAL_DEPTH = 2
HEADS = 8
DIM_HEAD = DIM // HEADS

# -- Training config --
BATCH_SIZE = 16
LR_STAGE1 = 5e-4
LR_PROJECTOR = 1e-3
LR_LORA = 2e-4
NUM_EPOCHS_STAGE1 = 10
NUM_EPOCHS_STAGE2 = 5

# -- LLM config --
LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"

print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name()}")
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"   VRAM: {vram:.1f} GB")
print(f"PASTIS24: {PASTIS24_DIR}")
print(f"   pickle24x24: {'OK' if PICKLE_DIR.is_dir() else 'NOT FOUND'}")
print(f"   fold-paths:  {'OK' if FOLDS_DIR.is_dir() else 'NOT FOUND'}")
print(f"{NUM_CLASSES} clases de cultivos")

Device: cuda
   GPU: NVIDIA GeForce RTX 5080
   VRAM: 16.6 GB
PASTIS-R:  /home/jsant16/proyecto-integrador/dataset/PASTIS-R  OK
Imagenes:  /home/jsant16/proyecto-integrador/train_data_v0  OK
18 clases de cultivos


## Carga de datos PASTIS24

Dataset pre-procesado del repo TSViT (DeepSatModels):
- Cada `.pickle` contiene `img` (T, 10, 24, 24), `doy` (T,), `labels` (3, 24, 24)
- Splits oficiales en `fold-paths/fold_{n}_paths.csv`
- Normalizacion: `img / 10000`, clip [0, 1]
- Canal de tiempo: `doy / 365` broadcast a (T, 1, 24, 24)
- Label: clase mayoritaria de `labels[0]` (semantic mask)

In [ ]:
# ============================================================
# Celda 3: Carga directa desde PASTIS24 pickles
# Formato: pickle24x24/{patchID}_{windowIdx}.pickle
#   img:    (T, 10, 24, 24) int16
#   doy:    (T,) int64
#   labels: (3, 24, 24) uint8 -> [semantic, instance, ...]
# ============================================================

def load_pickle_sample(path):
    with open(path, 'rb') as f:
        data = pickle.load(f)
    return data

def get_majority_class(semantic_mask):
    """Clase mayoritaria de la mascara semantica 24x24."""
    flat = semantic_mask.flatten()
    flat = flat[flat > 0]  # Ignorar background (0)
    flat = flat[flat <= 18]  # Solo clases validas (1-18)
    if len(flat) == 0:
        return 0  # Sin clase valida
    return int(np.bincount(flat).argmax())

def load_fold_paths(folds_dir, fold_num):
    """Lee paths de un fold CSV."""
    csv_path = folds_dir / f"fold_{fold_num}_paths.csv"
    if not csv_path.exists():
        return []
    df = pd.read_csv(csv_path, header=None)
    return df[0].tolist()

def load_all_data(pastis24_dir, folds_dir, train_folds=(1, 2, 3),
                  val_folds=(4,), test_folds=(5,), verbose=True):
    """
    Carga todos los datos usando los folds oficiales de PASTIS24.
    
    Default: Folds 1-3 train, Fold 4 val, Fold 5 test
    (mismo split que DeepSatModels para comparabilidad)
    """
    pickle_dir = pastis24_dir / "pickle24x24"
    
    splits = {
        'train': train_folds,
        'val': val_folds,
        'test': test_folds,
    }
    
    all_data = {}
    for split_name, fold_nums in splits.items():
        samples = []
        paths = []
        for fn in fold_nums:
            fold_paths = load_fold_paths(folds_dir, fn)
            paths.extend(fold_paths)
        
        skipped = 0
        iterator = tqdm(paths, desc=f"Cargando {split_name}") if verbose else paths
        for rel_path in iterator:
            full_path = pastis24_dir / rel_path.strip()
            if not full_path.exists():
                skipped += 1
                continue
            try:
                data = load_pickle_sample(full_path)
                img = data['img']      # (T, 10, 24, 24) int16
                doy = data['doy']      # (T,) int64
                labels = data['labels']  # (3, 24, 24) uint8
                
                # Clase mayoritaria de mascara semantica
                label = get_majority_class(labels[0])
                if label < 1 or label > 18:
                    skipped += 1
                    continue
                
                # Normalizar img: int16 / 10000 -> [0, 1]
                img_norm = np.clip(img.astype(np.float32) / 10000.0, 0, 1)
                
                # Canal de tiempo: doy / 365 -> (T, 1, 24, 24)
                T = img_norm.shape[0]
                time_ch = np.zeros((T, 1, IMG_RES, IMG_RES), dtype=np.float32)
                for t in range(T):
                    time_ch[t, 0, :, :] = doy[t] / 365.0
                
                # Concatenar: (T, 11, 24, 24)
                sits = np.concatenate([time_ch, img_norm], axis=1)
                
                samples.append({
                    'sits': sits,
                    'label': label,
                    'path': rel_path,
                })
            except Exception as e:
                skipped += 1
                if verbose:
                    tqdm.write(f"  Error {rel_path}: {e}")
                continue
        
        all_data[split_name] = samples
        if verbose:
            print(f"  {split_name}: {len(samples):,} samples ({skipped} skipped)")
    
    return all_data

# -- Ejecutar --
if PICKLE_DIR.is_dir() and FOLDS_DIR.is_dir():
    data_splits = load_all_data(PASTIS24_DIR, FOLDS_DIR)
    train_data = data_splits['train']
    val_data = data_splits['val']
    test_data = data_splits['test']
    
    # Resumen
    print(f"\nTotal: {len(train_data)+len(val_data)+len(test_data):,} samples")
    print(f"  Train: {len(train_data):,}")
    print(f"  Val:   {len(val_data):,}")
    print(f"  Test:  {len(test_data):,}")
    
    # Distribucion de clases en train
    counts = Counter(s['label'] for s in train_data)
    print(f"\nDistribucion (train):")
    for cls_id in sorted(counts.keys()):
        print(f"   {CLASSES.get(cls_id, '?')}: {counts[cls_id]:,}")
    
    all_parcels = train_data  # Para compatibilidad con celdas posteriores
else:
    print("PASTIS24 no encontrado. Verifica PASTIS24_DIR.")
    train_data, val_data, test_data = [], [], []
    all_parcels = []

Cache invalido o vacio (Ran out of input). Re-extraendo...


Extrayendo parcelas:   0%|          | 0/2468 [00:00<?, ?it/s]


79,681 parcelas extraidas (6270 muy pequenas)

Distribucion:
   Meadow: 29,421
   Soft_winter_wheat: 7,893
   Corn: 12,442
   Winter_barley: 2,646
   Winter_rapeseed: 1,722
   Spring_barley: 862
   Sunflower: 1,302
   Grapevine: 8,697
   Beet: 844
   Winter_triticale: 1,173
   Winter_durum_wheat: 1,616
   Fruits_vegetables_flowers: 2,326
   Potatoes: 494
   Leguminous_fodder: 2,950
   Soybeans: 1,179
   Orchard: 2,615
   Mixed_cereal: 823
   Sorghum: 676


In [ ]:
# ============================================================
# Celda 4: Dataset y DataLoaders
# ============================================================

class PASTIS24Dataset(Dataset):
    """Dataset para PASTIS24 pre-procesado."""
    
    def __init__(self, samples, max_seq_len=MAX_SEQ_LEN):
        self.samples = samples
        self.max_seq_len = max_seq_len
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        s = self.samples[idx]
        sits = s['sits']  # (T, 11, 24, 24) float32
        T = sits.shape[0]
        
        if T > self.max_seq_len:
            indices = np.linspace(0, T - 1, self.max_seq_len, dtype=int)
            sits = sits[indices]
            T = self.max_seq_len
        
        return {
            'sits': torch.from_numpy(sits).float(),
            'label': torch.tensor(s['label'] - 1).long(),  # 0-indexed
            'seq_len': torch.tensor(T).long(),
        }

def collate_fn(batch):
    """Collate con padding dinamico al max T del batch."""
    max_t = max(b['seq_len'].item() for b in batch)
    
    padded = []
    for b in batch:
        sits = b['sits']
        t = sits.shape[0]
        if t < max_t:
            pad = torch.zeros(max_t - t, 11, IMG_RES, IMG_RES)
            sits = torch.cat([sits, pad], dim=0)
        padded.append(sits[:max_t])
    
    return {
        'sits': torch.stack(padded),
        'label': torch.stack([b['label'] for b in batch]),
        'seq_len': torch.stack([b['seq_len'] for b in batch]),
    }

# -- Crear DataLoaders --
if train_data:
    train_ds = PASTIS24Dataset(train_data)
    val_ds = PASTIS24Dataset(val_data)
    test_ds = PASTIS24Dataset(test_data)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              collate_fn=collate_fn, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=collate_fn, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             collate_fn=collate_fn, num_workers=2, pin_memory=True)
    
    # Verificar un batch
    batch = next(iter(train_loader))
    print(f"Batch de prueba:")
    print(f"   sits: {batch['sits'].shape}")
    print(f"   labels: {batch['label'].shape} -> {batch['label'][:5].tolist()}")
    print(f"   seq_lens: {batch['seq_len'][:5].tolist()}")
else:
    print("Sin datos")

In [ ]:
# ============================================================
# Celda 5: Bloques Transformer
# Ref: TSViT (Tarasiou et al., CVPR 2023)
# ============================================================

class PreNorm(nn.Module):
    def __init__(self, dim, fn):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fn = fn
    def forward(self, x, **kw):
        return self.fn(self.norm(x), **kw)

class FeedForward(nn.Module):
    def __init__(self, dim, hidden_dim, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, hidden_dim), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, dim), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class Attention(nn.Module):
    def __init__(self, dim, heads=8, dim_head=16, dropout=0.0):
        super().__init__()
        inner = dim_head * heads
        self.heads = heads
        self.scale = dim_head ** -0.5
        self.to_qkv = nn.Linear(dim, inner * 3, bias=False)
        self.to_out = nn.Sequential(nn.Linear(inner, dim), nn.Dropout(dropout))
    
    def forward(self, x):
        b, n, _ = x.shape; h = self.heads
        qkv = self.to_qkv(x).chunk(3, dim=-1)
        q, k, v = map(lambda t: rearrange(t, 'b n (h d) -> b h n d', h=h), qkv)
        attn = (q @ k.transpose(-1, -2) * self.scale).softmax(dim=-1)
        out = rearrange(attn @ v, 'b h n d -> b n (h d)')
        return self.to_out(out)

class TransformerEncoder(nn.Module):
    def __init__(self, dim, depth, heads, dim_head, mlp_mult=4, dropout=0.0):
        super().__init__()
        self.layers = nn.ModuleList([nn.ModuleList([
            PreNorm(dim, Attention(dim, heads, dim_head, dropout)),
            PreNorm(dim, FeedForward(dim, dim * mlp_mult, dropout)),
        ]) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
    
    def forward(self, x):
        for attn, ff in self.layers:
            x = attn(x) + x
            x = ff(x) + x
        return self.norm(x)

print("✅ Transformer blocks definidos")

✅ Transformer blocks definidos


## 🧠 TSViT Encoder — Temporal → Spatial

Implementación fiel al paper CVPR 2023 (Fig. 4b):

1. **Tokenization**: Cada timestamp se divide en micro-patches (2×2) → 144 tokens espaciales
2. **Temporal Encoder**: Para cada ubicación espacial, procesa la serie temporal con K=18 CLS tokens prepended
3. **Spatial Encoder**: Para cada CLS token, procesa la distribución espacial
4. **Output**: K=18 CLS token features de dimensión d=128

In [ ]:
# ============================================================
# Celda 6: TSViT Encoder
# Ref: github.com/michaeltrs/DeepSatModels/models/TSViT/TSViTdense.py
# ============================================================

class TSViT(nn.Module):
    """
    Temporal-Spatial Vision Transformer para series temporales satelitales.

    Input: (B, T, C, H, W) donde C=11 (10 bandas + 1 time channel)
    Output: (B, K, dim) — K CLS tokens con features por clase
    """
    def __init__(self, img_res=24, patch_size=2, num_channels=11,
                 num_classes=18, dim=128, temporal_depth=6, spatial_depth=2,
                 heads=8, dim_head=16, dropout=0.1, max_seq_len=70):
        super().__init__()
        self.num_classes = num_classes
        self.dim = dim
        self.patch_size = patch_size
        
        NH = NW = img_res // patch_size  # 12
        self.NH = NH
        self.num_patches_space = NH * NW  # 144
        spectral_channels = num_channels - 1  # 10 (excluir time channel)
        patch_dim = spectral_channels * patch_size * patch_size  # 10 * 4 = 40
        
        # ━━━ Patch Embedding ━━━
        self.to_patch_embed = nn.Sequential(
            Rearrange('b t c (h p1) (w p2) -> (b h w) t (p1 p2 c)',
                      p1=patch_size, p2=patch_size),
            nn.LayerNorm(patch_dim),
            nn.Linear(patch_dim, dim),
            nn.LayerNorm(dim),
        )
        
        # ━━━ Temporal Position Encoding (day-of-year) ━━━
        self.temporal_pos_embed = nn.Embedding(366, dim)
        
        # ━━━ K CLS tokens para Temporal Encoder ━━━
        self.temporal_cls_tokens = nn.Parameter(
            torch.randn(1, num_classes, dim) * 0.02
        )
        
        # ━━━ Temporal Encoder ━━━
        self.temporal_encoder = TransformerEncoder(
            dim, temporal_depth, heads, dim_head, mlp_mult=4, dropout=dropout
        )
        
        # ━━━ Spatial Position Encoding ━━━
        self.spatial_pos_embed = nn.Parameter(
            torch.randn(1, self.num_patches_space + 1, dim) * 0.02  # +1 for global CLS
        )
        
        # ━━━ Global CLS token para Spatial Encoder ━━━
        self.spatial_cls_token = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        
        # ━━━ Spatial Encoder ━━━
        self.spatial_encoder = TransformerEncoder(
            dim, spatial_depth, heads, dim_head, mlp_mult=4, dropout=dropout
        )
    
    def _extract_time_and_spectral(self, x):
        """Separa canal de tiempo de bandas espectrales.
        Input: (B, T, 11, H, W) → spectral (B, T, 10, H, W), time_points (B, T)
        """
        time_feat = x[:, :, 0, :, :].mean(dim=(-1, -2))  # (B, T) — promedio del canal 0
        time_points = (time_feat * 365).long().clamp(0, 365)  # day-of-year
        spectral = x[:, :, 1:, :, :]  # (B, T, 10, H, W)
        return spectral, time_points
    
    def forward(self, x):
        """
        x: (B, T, 11, H, W)
        returns: (B, K, dim) — K CLS token features
        """
        B, T, C, H, W = x.shape
        K = self.num_classes
        NH = self.NH
        
        # ── 1. Separar tiempo y espectral ──
        spectral, time_points = self._extract_time_and_spectral(x)  
        # spectral: (B, T, 10, H, W), time_points: (B, T)
        
        # ── 2. Patch embedding ──
        tokens = self.to_patch_embed(spectral)  # (B*NH*NW, T, dim)
        
        # ── 3. Temporal position encoding ──
        # time_points: (B, T) → necesitamos (B*NH*NW, T)
        tp_expanded = time_points.unsqueeze(1).expand(B, NH * NH, T)  # (B, NH*NW, T)
        tp_expanded = tp_expanded.reshape(B * NH * NH, T)
        temporal_pos = self.temporal_pos_embed(tp_expanded)  # (B*NH*NW, T, dim)
        tokens = tokens + temporal_pos
        
        # ── 4. Prepend K CLS tokens ──
        cls_tokens = self.temporal_cls_tokens.expand(B * NH * NH, -1, -1)  # (B*NH*NW, K, dim)
        tokens = torch.cat([cls_tokens, tokens], dim=1)  # (B*NH*NW, K+T, dim)
        
        # ── 5. Temporal Encoder ──
        tokens = self.temporal_encoder(tokens)  # (B*NH*NW, K+T, dim)
        
        # ── 6. Extraer solo CLS tokens ──
        cls_out = tokens[:, :K, :]  # (B*NH*NW, K, dim)
        
        # ── 7. Reshape para Spatial Encoder ──
        # (B*NH*NW, K, dim) → (B*K, NH*NW, dim)
        cls_out = cls_out.reshape(B, NH * NH, K, self.dim)
        cls_out = cls_out.permute(0, 2, 1, 3)  # (B, K, NH*NW, dim)
        cls_out = cls_out.reshape(B * K, NH * NH, self.dim)  # (B*K, NH*NW, dim)
        
        # ── 8. Prepend global CLS + spatial pos ──
        global_cls = self.spatial_cls_token.expand(B * K, -1, -1)  # (B*K, 1, dim)
        spatial_tokens = torch.cat([global_cls, cls_out], dim=1)  # (B*K, 1+NH*NW, dim)
        spatial_tokens = spatial_tokens + self.spatial_pos_embed[:, :1 + NH * NH, :]
        
        # ── 9. Spatial Encoder ──
        spatial_tokens = self.spatial_encoder(spatial_tokens)  # (B*K, 1+NH*NW, dim)
        
        # ── 10. Extraer global CLS token ──
        global_out = spatial_tokens[:, 0, :]  # (B*K, dim)
        global_out = global_out.reshape(B, K, self.dim)  # (B, K, dim)
        
        return global_out  # (B, K=18, dim=128)

# ── Instanciar y verificar ──
tsvit = TSViT(
    img_res=IMG_RES, patch_size=PATCH_SIZE, num_channels=11,
    num_classes=NUM_CLASSES, dim=DIM,
    temporal_depth=TEMPORAL_DEPTH, spatial_depth=SPATIAL_DEPTH,
    heads=HEADS, dim_head=DIM_HEAD, dropout=0.1
).to(DEVICE)

# Test
with torch.no_grad():
    dummy = torch.randn(2, 30, 11, IMG_RES, IMG_RES).to(DEVICE)
    out = tsvit(dummy)
    print(f"✅ TSViT: input {dummy.shape} → output {out.shape}")
    
n_params = sum(p.numel() for p in tsvit.parameters())
print(f"   Parámetros: {n_params:,} ({n_params/1e6:.1f}M)")
print(f"   VRAM: ~{n_params * 4 / 1e9:.3f} GB (fp32)")

IndentationError: unexpected indent (566784863.py, line 7)

In [ ]:
# ============================================================
# Celda 7: Focal Loss (manejo de desbalance)
# Ref: Lin et al. "Focal Loss for Dense Object Detection" (2017)
#      TSViT paper usa Focal Loss para PASTIS
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction
    
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        p_t = torch.exp(-ce)
        focal = ((1 - p_t) ** self.gamma) * ce
        if self.reduction == 'mean':
            return focal.mean()
        return focal

# Calcular pesos por clase (inverse frequency)
if train_data:
    counts = Counter(s['label'] - 1 for s in train_data)  # 0-indexed
    total = sum(counts.values())
    weights = torch.zeros(NUM_CLASSES)
    for c in range(NUM_CLASSES):
        freq = counts.get(c, 1)
        weights[c] = total / (NUM_CLASSES * freq)
    weights = weights / weights.sum() * NUM_CLASSES
    weights = weights.to(DEVICE)
    print("⚖️  Pesos por clase (top 5):")
    top5 = weights.argsort(descending=True)[:5]
    for idx in top5:
        print(f"   {CLASSES[idx.item()+1]}: {weights[idx]:.2f}")
else:
    weights = None

print("✅ Focal Loss definido")

⚖️  Pesos por clase (top 5):
   Potatoes: 2.83
   Sorghum: 2.22
   Mixed_cereal: 1.69
   Beet: 1.60
   Spring_barley: 1.54
✅ Focal Loss definido


## 🏋️ Stage 1: Entrenar TSViT standalone

**Objetivo**: Que el encoder aprenda features temporales-espaciales discriminativas.

El paper reporta ~83% OA en PASTIS con datos completos. Aquí entrenamos el encoder
con una MLP head temporal antes de conectar al LLM.

In [ ]:
# ============================================================
# Celda 8: Stage 1 — TSViT standalone con MLP head
# ============================================================

class TSViTClassifier(nn.Module):
    """TSViT + classification head para Stage 1."""
    def __init__(self, tsvit, num_classes=18):
        super().__init__()
        self.tsvit = tsvit
        # K CLS tokens → clasificación por votación diagonal
        # Cada CLS token k predice logit para su clase
        # Output: logit[k] = CLS_k · W_k (escalar por clase)
        self.head = nn.Linear(tsvit.dim, 1)  # cada CLS → 1 logit
    
    def forward(self, x):
        cls_tokens = self.tsvit(x)  # (B, K, dim)
        logits = self.head(cls_tokens).squeeze(-1)  # (B, K)
        return logits

classifier = TSViTClassifier(tsvit, NUM_CLASSES).to(DEVICE)

focal_loss = FocalLoss(gamma=2.0, weight=weights)
optimizer_s1 = torch.optim.AdamW(classifier.parameters(), lr=LR_STAGE1, weight_decay=0.01)
scheduler_s1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_s1, T_max=NUM_EPOCHS_STAGE1)

print(f"🏋️ Stage 1: TSViT Classifier")
print(f"   Epochs: {NUM_EPOCHS_STAGE1}")
print(f"   LR: {LR_STAGE1}")
print(f"   Params entrenables: {sum(p.numel() for p in classifier.parameters() if p.requires_grad):,}")

🏋️ Stage 1: TSViT Classifier
   Epochs: 10
   LR: 0.0005
   Params entrenables: 1,657,169


In [ ]:
# ============================================================
# Celda 9: Training loop Stage 1
# ============================================================

def train_epoch_s1(model, loader, optimizer, loss_fn, device, max_batches=None):
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    pbar = tqdm(loader, desc="Train", leave=False)
    for i, batch in enumerate(pbar):
        if max_batches and i >= max_batches:
            break
        
        sits = batch['sits'].to(device)
        labels = batch['label'].to(device)
        
        logits = model(sits)  # (B, 18)
        loss = loss_fn(logits, labels)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item() * labels.size(0)
        
        pbar.set_postfix(loss=f"{loss.item():.3f}", acc=f"{correct/total:.1%}")
    
    return total_loss / max(total, 1), correct / max(total, 1)

@torch.no_grad()
def eval_epoch_s1(model, loader, loss_fn, device, max_batches=None):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_preds, all_labels = [], []
    
    for i, batch in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        
        sits = batch['sits'].to(device)
        labels = batch['label'].to(device)
        
        logits = model(sits)
        loss = loss_fn(logits, labels)
        
        preds = logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        total_loss += loss.item() * labels.size(0)
        
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    
    acc = correct / max(total, 1)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / max(total, 1), acc, f1

# ── Entrenar ──
history_s1 = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}
best_val_acc = 0

print(f"\n{'='*60}")
print(f"STAGE 1: Entrenando TSViT Encoder")
print(f"{'='*60}")

for epoch in range(NUM_EPOCHS_STAGE1):
    t_loss, t_acc = train_epoch_s1(classifier, train_loader, optimizer_s1,
                                    focal_loss, DEVICE)
    v_loss, v_acc, v_f1 = eval_epoch_s1(classifier, val_loader, focal_loss, DEVICE)
    scheduler_s1.step()
    
    history_s1['train_loss'].append(t_loss)
    history_s1['train_acc'].append(t_acc)
    history_s1['val_loss'].append(v_loss)
    history_s1['val_acc'].append(v_acc)
    history_s1['val_f1'].append(v_f1)
    
    improved = ""
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(tsvit.state_dict(), "tsvit_best.pt")
        improved = " ⭐ best"
    
    lr_now = scheduler_s1.get_last_lr()[0]
    print(f"  Epoch {epoch+1:2d}/{NUM_EPOCHS_STAGE1} | "
          f"Train: loss={t_loss:.3f} acc={t_acc:.1%} | "
          f"Val: loss={v_loss:.3f} acc={v_acc:.1%} F1={v_f1:.3f} | "
          f"LR={lr_now:.1e}{improved}")

print(f"\n✅ Best val accuracy: {best_val_acc:.1%}")


STAGE 1: Entrenando TSViT Encoder


Train:   0%|          | 0/3499 [00:00<?, ?it/s]

  Epoch  1/10 | Train: loss=0.342 acc=17.8% | Val: loss=0.223 acc=26.9% F1=0.353 | LR=4.9e-04 ⭐ best


Train:   0%|          | 0/3499 [00:00<?, ?it/s]

  Epoch  2/10 | Train: loss=0.183 acc=34.5% | Val: loss=0.164 acc=30.9% F1=0.412 | LR=4.5e-04 ⭐ best


...

Train:   0%|          | 0/3499 [00:00<?, ?it/s]

  Epoch  8/10 | Train: loss=0.077 acc=51.5% | Val: loss=0.131 acc=45.9% F1=0.540 | LR=4.8e-05


Train:   0%|          | 0/3499 [00:00<?, ?it/s]

  Epoch  9/10 | Train: loss=0.069 acc=53.2% | Val: loss=0.125 acc=49.6% F1=0.536 | LR=1.2e-05 ⭐ best


Train:   0%|          | 0/3499 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# Celda 10: Curvas de entrenamiento Stage 1
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history_s1['train_loss'], label='Train', marker='o', markersize=3)
axes[0].plot(history_s1['val_loss'], label='Val', marker='s', markersize=3)
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')

axes[1].plot(history_s1['train_acc'], label='Train', marker='o', markersize=3)
axes[1].plot(history_s1['val_acc'], label='Val', marker='s', markersize=3)
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('Epoch')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

axes[2].plot(history_s1['val_f1'], label='Val F1', marker='d', markersize=3, color='green')
axes[2].set_title('F1 Macro'); axes[2].legend(); axes[2].set_xlabel('Epoch')

plt.suptitle('Stage 1: TSViT Standalone Training', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 🔗 Stage 2: Conectar TSViT → Projector → LLM (LoRA)

**Componentes:**
1. **TSViT** (frozen) — usa los pesos de Stage 1
2. **Projector** (Adapter, trainable) — MLP 2 capas: `dim → llm_dim → llm_dim`
3. **LLM** (Qwen2.5-3B, 4-bit + LoRA) — genera texto

**Prompt template:**
```
<|im_start|>system
You classify crops from satellite time series features.<|im_end|>
<|im_start|>user
[VISUAL TOKENS FROM TSVIT]
What crop is shown in this satellite image time series?<|im_end|>
<|im_start|>assistant
{crop_name}<|im_end|>
```

In [ ]:
# ============================================================
# Celda 11: Projector (Adapter) y modelo TSViT-LLM
# ============================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

class Projector(nn.Module):
    """MLP Projector: TSViT dim → LLM dim (LLaVA pattern)."""
    def __init__(self, tsvit_dim, llm_dim):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(tsvit_dim, llm_dim),
            nn.GELU(),
            nn.Linear(llm_dim, llm_dim),
        )
    def forward(self, x):
        return self.mlp(x)  # (B, K, llm_dim)

class TSViTLLM(nn.Module):
    """
    TSViT encoder (frozen) → Projector (trainable) → LLM decoder (LoRA).
    
    Entrenamiento:
        - visual_tokens = projector(tsvit(sits))
        - text_tokens = tokenizer(prompt + answer)
        - concat [visual_tokens, text_tokens]
        - LLM forward con causal LM loss (solo en answer tokens)
    """
    def __init__(self, tsvit, llm_name, tsvit_dim=128):
        super().__init__()
        
        # ── TSViT encoder (FROZEN) ──
        self.tsvit = tsvit
        for p in self.tsvit.parameters():
            p.requires_grad = False
        self.tsvit.eval()
        
        # ── LLM (4-bit + LoRA) ──
        print(f"  Cargando {llm_name} en 4-bit...")
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
        self.llm = AutoModelForCausalLM.from_pretrained(
            llm_name, quantization_config=bnb_config,
            device_map="auto", torch_dtype=torch.float16
        )
        self.tokenizer = AutoTokenizer.from_pretrained(llm_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # ── LoRA ──
        print("  Aplicando LoRA...")
        lora_config = LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            r=16, lora_alpha=32, lora_dropout=0.05,
            target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
        )
        self.llm = get_peft_model(self.llm, lora_config)
        
        # ── Projector ──
        llm_dim = self.llm.config.hidden_size
        self.projector = Projector(tsvit_dim, llm_dim).to(self.llm.device)
        
        print(f"  ✅ LLM hidden_size: {llm_dim}")
        print(f"  ✅ LoRA params: {self.llm.print_trainable_parameters()}")
    
    def _get_text_embeddings(self, text_list):
        """Tokeniza y obtiene embeddings de texto."""
        tokens = self.tokenizer(text_list, return_tensors="pt", padding=True,
                                truncation=True, max_length=64)
        tokens = {k: v.to(self.llm.device) for k, v in tokens.items()}
        embeds = self.llm.get_input_embeddings()(tokens['input_ids'])
        return embeds, tokens['attention_mask'], tokens['input_ids']
    
    def forward(self, sits, prompt_text, answer_text):
        """
        sits: (B, T, 11, H, W)
        prompt_text: list[str] — prompts
        answer_text: list[str] — respuestas (crop names)
        
        Returns: loss (scalar)
        """
        B = sits.size(0)
        
        # ── 1. TSViT features (frozen) ──
        with torch.no_grad():
            cls_features = self.tsvit(sits)  # (B, K, tsvit_dim)
        
        # ── 2. Project a LLM space ──
        visual_tokens = self.projector(cls_features.to(self.projector.mlp[0].weight.dtype))
        visual_tokens = visual_tokens.to(self.llm.device)  # (B, K, llm_dim)
        
        # ── 3. Tokenizar prompt y answer ──
        prompt_embeds, prompt_mask, _ = self._get_text_embeddings(prompt_text)
        answer_embeds, answer_mask, answer_ids = self._get_text_embeddings(answer_text)
        
        # ── 4. Concatenar: [visual | prompt | answer] ──
        all_embeds = torch.cat([visual_tokens, prompt_embeds, answer_embeds], dim=1)
        
        # Attention mask
        visual_mask = torch.ones(B, visual_tokens.size(1), device=self.llm.device)
        all_mask = torch.cat([visual_mask, prompt_mask, answer_mask], dim=1)
        
        # ── 5. Labels: -100 para visual+prompt, real para answer ──
        ignore_len = visual_tokens.size(1) + prompt_embeds.size(1)
        ignore_labels = torch.full((B, ignore_len), -100, device=self.llm.device, dtype=torch.long)
        
        # Shift labels para causal LM
        answer_labels = answer_ids.clone()
        answer_labels[answer_mask == 0] = -100
        
        labels = torch.cat([ignore_labels, answer_labels], dim=1)
        
        # ── 6. LLM forward ──
        outputs = self.llm(inputs_embeds=all_embeds, attention_mask=all_mask, labels=labels)
        
        return outputs.loss
    
    @torch.no_grad()
    def generate(self, sits, prompt_text, max_new_tokens=32):
        """Genera predicción de cultivo."""
        self.tsvit.eval()
        B = sits.size(0)
        
        # Visual tokens
        cls_features = self.tsvit(sits)
        visual_tokens = self.projector(cls_features.to(self.projector.mlp[0].weight.dtype))
        visual_tokens = visual_tokens.to(self.llm.device)
        
        # Prompt embeddings
        prompt_embeds, prompt_mask, _ = self._get_text_embeddings(prompt_text)
        
        # Concat
        input_embeds = torch.cat([visual_tokens, prompt_embeds], dim=1)
        visual_mask = torch.ones(B, visual_tokens.size(1), device=self.llm.device)
        attn_mask = torch.cat([visual_mask, prompt_mask], dim=1)
        
        # Generate
        outputs = self.llm.generate(
            inputs_embeds=input_embeds,
            attention_mask=attn_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=self.tokenizer.pad_token_id,
        )
        
        # Decode
        texts = self.tokenizer.batch_decode(outputs, skip_special_tokens=True)
        return texts

In [ ]:
# ============================================================
# Celda 12: Instanciar TSViT-LLM
# ============================================================

# Cargar mejores pesos de Stage 1
tsvit.load_state_dict(torch.load("tsvit_best.pt", map_location=DEVICE))
print("✅ TSViT cargado con pesos de Stage 1")

# Limpiar VRAM
del classifier, optimizer_s1, scheduler_s1
gc.collect()
torch.cuda.empty_cache() if DEVICE == "cuda" else None

# Crear modelo completo
model = TSViTLLM(tsvit, LLM_NAME, tsvit_dim=DIM)

# Prompt template
PROMPT = "What crop is shown in this satellite image time series? Answer with just the crop name."

# Answer mapping (label_id 0-indexed → text)
def make_answer(label_id):
    name = CLASSES[label_id + 1]  # 1-indexed
    return name.replace("_", " ").lower()

# VRAM check
if DEVICE == "cuda":
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"\n📊 VRAM: {allocated:.1f} GB allocated / {reserved:.1f} GB reserved")

In [ ]:
# ============================================================
# Celda 13: Stage 2 — Entrenar Projector + LoRA
# ============================================================

# Solo entrenar projector + LoRA
trainable_params = [
    {'params': model.projector.parameters(), 'lr': LR_PROJECTOR},
    {'params': [p for p in model.llm.parameters() if p.requires_grad], 'lr': LR_LORA},
]
optimizer_s2 = torch.optim.AdamW(trainable_params, weight_decay=0.01)
scheduler_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_s2, T_max=NUM_EPOCHS_STAGE2)

n_proj = sum(p.numel() for p in model.projector.parameters())
n_lora = sum(p.numel() for p in model.llm.parameters() if p.requires_grad)
print(f"🔧 Params entrenables:")
print(f"   Projector: {n_proj:,}")
print(f"   LoRA: {n_lora:,}")
print(f"   Total: {n_proj + n_lora:,}")

def train_epoch_s2(model, loader, optimizer, device, max_batches=None):
    model.projector.train()
    model.llm.train()
    model.tsvit.eval()
    
    total_loss, n_batches = 0, 0
    pbar = tqdm(loader, desc="Stage2-Train", leave=False)
    
    for i, batch in enumerate(pbar):
        if max_batches and i >= max_batches:
            break
        
        sits = batch['sits'].to(device)
        labels = batch['label']  # 0-indexed
        
        prompts = [PROMPT] * sits.size(0)
        answers = [make_answer(l.item()) for l in labels]
        
        loss = model(sits, prompts, answers)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(model.projector.parameters()) + 
            [p for p in model.llm.parameters() if p.requires_grad],
            max_norm=1.0
        )
        optimizer.step()
        
        total_loss += loss.item()
        n_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.3f}")
    
    return total_loss / max(n_batches, 1)

print(f"\n{'='*60}")
print(f"STAGE 2: Entrenando Projector + LoRA")
print(f"{'='*60}")

history_s2 = {'train_loss': []}
for epoch in range(NUM_EPOCHS_STAGE2):
    loss = train_epoch_s2(model, train_loader, optimizer_s2, DEVICE)
    scheduler_s2.step()
    history_s2['train_loss'].append(loss)
    
    lr_now = scheduler_s2.get_last_lr()[0]
    print(f"  Epoch {epoch+1}/{NUM_EPOCHS_STAGE2} | Loss={loss:.4f} | LR={lr_now:.1e}")

print("\n✅ Stage 2 completado")

In [ ]:
# ============================================================
# Celda 14: Parser robusto de nombres de cultivos
# ============================================================
import difflib

# Aliases para fuzzy matching
CROP_ALIASES = {
    'meadow': 0, 'prairie': 0, 'grassland': 0, 'pasture': 0,
    'soft winter wheat': 1, 'winter wheat': 1, 'wheat': 1, 'soft wheat': 1,
    'corn': 2, 'maize': 2,
    'winter barley': 3, 'barley': 3,
    'winter rapeseed': 4, 'rapeseed': 4, 'canola': 4, 'rape': 4,
    'spring barley': 5,
    'sunflower': 6, 'sunflowers': 6,
    'grapevine': 7, 'grape': 7, 'vineyard': 7, 'vine': 7,
    'beet': 8, 'sugar beet': 8, 'beetroot': 8,
    'winter triticale': 9, 'triticale': 9,
    'winter durum wheat': 10, 'durum wheat': 10, 'durum': 10,
    'fruits vegetables flowers': 11, 'fruits': 11, 'vegetables': 11, 'flowers': 11,
    'potatoes': 12, 'potato': 12,
    'leguminous fodder': 13, 'alfalfa': 13, 'clover': 13, 'fodder': 13,
    'soybeans': 14, 'soybean': 14, 'soy': 14,
    'orchard': 15, 'fruit trees': 15,
    'mixed cereal': 16, 'mixed cereals': 16,
    'sorghum': 17,
}

def parse_crop_name(text):
    """Parsea la respuesta del LLM a un label_id (0-indexed)."""
    text_clean = text.lower().strip().rstrip('.').strip()
    
    # 1. Match exacto
    if text_clean in CROP_ALIASES:
        return CROP_ALIASES[text_clean]
    
    # 2. Match parcial (el texto contiene el alias)
    for alias, label in sorted(CROP_ALIASES.items(), key=lambda x: -len(x[0])):
        if alias in text_clean:
            return label
    
    # 3. Fuzzy matching
    matches = difflib.get_close_matches(text_clean, CROP_ALIASES.keys(), n=1, cutoff=0.6)
    if matches:
        return CROP_ALIASES[matches[0]]
    
    return -1  # No match

# Test
for test in ["soft winter wheat", "Corn.", "It appears to be sunflower", "RAPESEED", "unknown crop"]:
    result = parse_crop_name(test)
    label_name = CLASSES.get(result + 1, "UNKNOWN") if result >= 0 else "UNKNOWN"
    print(f"  '{test}' → {result} ({label_name})")

In [ ]:
# ============================================================
# Celda 15: Evaluación en test set
# ============================================================

@torch.no_grad()
def evaluate_tsvit_llm(model, loader, device, max_batches=None):
    model.tsvit.eval()
    model.projector.eval()
    model.llm.eval()
    
    all_preds, all_labels, all_texts = [], [], []
    
    pbar = tqdm(loader, desc="Evaluando")
    for i, batch in enumerate(pbar):
        if max_batches and i >= max_batches:
            break
        
        sits = batch['sits'].to(device)
        labels = batch['label']  # 0-indexed
        
        prompts = [PROMPT] * sits.size(0)
        
        # Generar
        generated = model.generate(sits, prompts, max_new_tokens=32)
        
        for text, label in zip(generated, labels):
            pred = parse_crop_name(text)
            all_preds.append(pred)
            all_labels.append(label.item())
            all_texts.append(text)
    
    # Métricas
    valid = [(p, l) for p, l in zip(all_preds, all_labels) if p >= 0]
    if valid:
        preds_v, labels_v = zip(*valid)
        acc = accuracy_score(labels_v, preds_v)
        f1 = f1_score(labels_v, preds_v, average='macro', zero_division=0)
    else:
        acc, f1 = 0, 0
    
    parse_rate = sum(1 for p in all_preds if p >= 0) / len(all_preds)
    
    return {
        'accuracy': acc,
        'f1_macro': f1,
        'parse_rate': parse_rate,
        'predictions': all_preds,
        'labels': all_labels,
        'texts': all_texts,
    }

# ── Evaluar ──
print("\n" + "="*60)
print("EVALUACIÓN EN TEST SET")
print("="*60)

results = evaluate_tsvit_llm(model, test_loader, DEVICE)

print(f"\n📊 Resultados TSViT + LLM:")
print(f"   Accuracy: {results['accuracy']:.1%}")
print(f"   F1 Macro: {results['f1_macro']:.3f}")
print(f"   Parse rate: {results['parse_rate']:.1%}")

# Ejemplos
print(f"\n📝 Ejemplos de predicciones:")
for i in range(min(10, len(results['texts']))):
    gt = CLASSES.get(results['labels'][i] + 1, "?")
    pred_id = results['predictions'][i]
    pred_name = CLASSES.get(pred_id + 1, "UNKNOWN") if pred_id >= 0 else "PARSE_FAIL"
    status = "✅" if results['predictions'][i] == results['labels'][i] else "❌"
    print(f"   {status} GT={gt:25s} | Pred={pred_name:25s} | Raw: {results['texts'][i][:50]}")

In [ ]:
# ============================================================
# Celda 16: Reporte detallado + Matriz de confusión
# ============================================================

# Classification report
valid_mask = [i for i, p in enumerate(results['predictions']) if p >= 0]
if valid_mask:
    y_true = [results['labels'][i] for i in valid_mask]
    y_pred = [results['predictions'][i] for i in valid_mask]
    
    present_classes = sorted(set(y_true + y_pred))
    target_names = [CLASSES.get(c + 1, f"cls_{c}") for c in present_classes]
    
    print("\n📋 Classification Report:")
    print(classification_report(y_true, y_pred, labels=present_classes,
                                target_names=target_names, zero_division=0))
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=present_classes)
    
    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    ax.set_xticks(range(len(target_names)))
    ax.set_yticks(range(len(target_names)))
    ax.set_xticklabels(target_names, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(target_names, fontsize=8)
    
    # Anotaciones
    thresh = cm.max() / 2.
    for i in range(len(target_names)):
        for j in range(len(target_names)):
            ax.text(j, i, f"{cm[i, j]}", ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black", fontsize=7)
    
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title('TSViT + LLM — Confusion Matrix')
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout(); plt.show()
else:
    print("⚠️ No hay predicciones válidas para reporte")

In [ ]:
# ============================================================
# Celda 17: Comparación Stage 1 vs Stage 2
# ============================================================

# Stage 1 results (del entrenamiento anterior)
s1_best_acc = best_val_acc
s1_test_loss, s1_test_acc, s1_test_f1 = 0, 0, 0

# Re-evaluar Stage 1 en test si tenemos el clasificador
try:
    classifier_eval = TSViTClassifier(tsvit, NUM_CLASSES).to(DEVICE)
    classifier_eval.tsvit.load_state_dict(torch.load("tsvit_best.pt", map_location=DEVICE))
    _, s1_test_acc, s1_test_f1 = eval_epoch_s1(classifier_eval, test_loader, focal_loss, DEVICE)
    del classifier_eval
except Exception:
    s1_test_acc = s1_best_acc
    s1_test_f1 = max(history_s1['val_f1']) if history_s1['val_f1'] else 0

print(f"\n{'='*60}")
print(f"COMPARACIÓN DE RESULTADOS")
print(f"{'='*60}")
print(f"")
print(f"{'Métrica':<20} {'TSViT Standalone':<20} {'TSViT + LLM':<20}")
print(f"{'-'*60}")
print(f"{'Test Accuracy':<20} {s1_test_acc:<20.1%} {results['accuracy']:<20.1%}")
print(f"{'Test F1 Macro':<20} {s1_test_f1:<20.3f} {results['f1_macro']:<20.3f}")
print(f"{'Output type':<20} {'Class index':<20} {'Natural language':<20}")
print(f"{'Flexible input':<20} {'No (fixed SITS)':<20} {'No (fixed SITS)':<20}")
print(f"{'Interpretable':<20} {'No':<20} {'Yes (text output)':<20}")
print(f"")
print(f"💡 Nota: con todos los datos (~60K muestras) y epochs completos,")
print(f"   TSViT standalone debería alcanzar ~83% OA según el paper (CVPR 2023).")

In [ ]:
# ============================================================
# Celda 18: Guardar checkpoints
# ============================================================

# TSViT encoder weights
torch.save(tsvit.state_dict(), "tsvit_encoder.pt")
print("💾 tsvit_encoder.pt guardado")

# Projector weights
torch.save(model.projector.state_dict(), "projector.pt")
print("💾 projector.pt guardado")

# LoRA adapters
model.llm.save_pretrained("lora_weights/")
print("💾 lora_weights/ guardado")

# Training history
import json as _json
history = {
    'stage1': history_s1,
    'stage2': history_s2,
    'test_results': {
        'accuracy': results['accuracy'],
        'f1_macro': results['f1_macro'],
        'parse_rate': results['parse_rate'],
    }
}
with open("training_history.json", "w") as f:
    _json.dump(history, f, indent=2)
print("💾 training_history.json guardado")

print(f"\n✅ Todo guardado. Para cargar:")
print(f"   tsvit.load_state_dict(torch.load('tsvit_encoder.pt'))")
print(f"   model.projector.load_state_dict(torch.load('projector.pt'))")
print(f"   model.llm = PeftModel.from_pretrained(base_model, 'lora_weights/')")

## Resumen

### Arquitectura implementada
- **Encoder**: TSViT (Temporal-Spatial ViT, CVPR 2023) — 6 capas temporales + 2 espaciales
- **Projector**: MLP 2 capas (128 -> LLM dim) — patron LLaVA
- **Decoder**: Qwen2.5-3B-Instruct (4-bit + LoRA r=16)

### Dataset: PASTIS24
- Formato pre-procesado del repo DeepSatModels
- `pickle24x24/` con ~60K ventanas 24x24
- 5-fold split oficial en `fold-paths/`
- Folds 1-3: train, Fold 4: val, Fold 5: test

### Entrenamiento en 2 etapas
1. Stage 1: TSViT standalone con Focal Loss (encoder aprende features)
2. Stage 2: Projector + LoRA, TSViT frozen (LLM genera clasificacion en texto)

### Fuentes verificadas
| Componente | Referencia |
|-----------|-----------|
| TSViT | Tarasiou et al., CVPR 2023 |
| PASTIS / PASTIS24 | Garnot & Landrieu, 2022 |
| Projector MLP | Liu et al., LLaVA, NeurIPS 2023 |
| LoRA | Hu et al., ICLR 2022 |
| Focal Loss | Lin et al., ICCV 2017 |